In [0]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame, Column

#### Bronze 

##### Bronze ingestion

In [0]:
def ingest_bronze_table(source:str, src_table:str, bronze_table:str) -> None:
    src_df = spark.read.table(src_table)
    bronze_df = (src_df
    .withColumn('_batch_id', F.lit('week03_inital_load'))
    .withColumn('_ingested_at', F.current_timestamp())
    .withColumn('_source', F.lit(source))
    )
    bronze_df.write.mode('overwrite').format('delta').saveAsTable(bronze_table)

#### Silver

##### 1. Data standarization 

In [0]:
def standarize_transactions(df:DataFrame) -> DataFrame :
    cols = ['transactionID', 'customerID', 'franchiseID', 'dateTime', 'product', 'quantity', 'unitPrice', 'totalPrice']
    return (
        df.select(cols)
        .withColumn('transactionID', F.col('transactionID').cast('string'))
        .withColumn('customerID', F.col('customerID').cast('string'))
        .withColumn('franchiseID', F.col('franchiseID').cast('string'))
        .withColumn('transaction_date', F.col('dateTime').cast('date'))
        .withColumn('unitPrice', F.col('unitPrice').cast('decimal(10,2)'))
        .withColumn('totalPrice', F.col('totalPrice').cast('decimal(10,2)'))
        .withColumn('calculated_total', F.col('unitPrice')*F.col('quantity').cast('decimal(10,2)'))
    )
        
def standarize_customers(df:DataFrame) -> DataFrame :
    cols = ['customerID', 'first_name', 'last_name', 'city', 'state', 'country', 'continent', 'gender']
    return (
        df.select(cols)
        .withColumn('customerID', F.col('customerID').cast('string'))
    )

def standarize_franchises(df:DataFrame) -> DataFrame :
    cols = ['franchiseID', 'name', 'city', 'district', 'country', 'size']
    return (
        df.select(cols)
        .withColumn('franchiseID', F.col('franchiseID').cast('string'))
        .withColumnRenamed('name', 'franchiseName')
    )

##### 2. Data deduplication

In [0]:
def check_global_quality(df:DataFrame, key:str) -> DataFrame :
    global_quality = {
        'null_key': (df.filter(F.col(key).isNull()).count()),
        'identical_rows':(
            df.groupBy(*df.columns).count().filter(F.col('count') > 1).count()
        )
    }
    print(global_quality)

    if global_quality['identical_rows'] > 0:
        print('There are duplicate rows in the dataset')

        dup_rows = df.groupBy(*df.columns).count().filter(F.col('count') > 1)

        df_dedup = df.dropDuplicates()
        dedup_rows = df_dedup.count()

        input_rows = df.count()
        
        removed_dup_rows = dup_rows.select(F.sum(F.col('count') -1).alias('removed')).first()['removed']

        assert input_rows == dedup_rows + removed_dup_rows
    else:
        df_dedup = df

    # Check if there are different rows sharing the same key after deduplication the identical rows
    dup_keys_rows = df_dedup.groupBy(key).count().filter(F.col('count') > 1)
    if dup_keys_rows.count() > 0:
        raise ValueError(f'There are different rows sharing the same {key}')
    else:
        print(f'No diffrent rows sharing the same {key} after checking the identical duplicates.')

    return df_dedup

##### 3. Data validation 

In [0]:
def is_missing(col_name:str) -> Column:
    return (
        F.when(
            F.col(col_name).isNull() | (F.trim(F.col(col_name)) == ''),
            F.lit(f'missing_{col_name}')
        )
    )

In [0]:
def validate_customers(df):
    return(
        df
        .withColumn(
            'dq_reason',
            F.coalesce(
                is_missing('customerID'),
                is_missing('first_name'),
                is_missing('last_name'),
                is_missing('city'),
                is_missing('state'),
                is_missing('country'),

                F.when(
                    F.col('continent').isNotNull() & 
                    ~F.initcap(F.col('continent')).isin(
                        'Africa',
                        'Asia',
                        'Europe',
                        'North America',
                        'South America',
                        'Oceania',
                        'Antarctica'
                    ),
                    F.lit('invalid_continent')
                )
                .when(
                    F.col('gender').isNotNull() & 
                    ~F.lower(F.col('gender')).isin('female','male'),
                    F.lit('invalid_gender')
                )
            )
        )
        .withColumn(
            'dq_status',
            F.when(
                F.col('dq_reason').isNull(),
                F.lit('VALID')
            ).otherwise(F.lit('INVALID'))
        )
        .withColumn(
            'checked_at',
            F.current_timestamp()
        )
    )


In [0]:
def validate_franchises(df):
    return(
        df
        .withColumn(
            'dq_reason',
           F.coalesce(
                is_missing('franchiseID'),
                is_missing('franchiseName'),
                is_missing('city'),
                is_missing('district'),
                is_missing('country'),
                F.when(
                    F.col('size').isNotNull() & 
                    ~F.col('size').isin('S', 'M', 'L', 'XL', 'XXL'),
                    F.lit('invalid_size')
                )
            )   
        )
        .withColumn(
            'dq_status',
            F.when(
                F.col('dq_reason').isNull(),
                F.lit('VALID')
            ).otherwise(F.lit('INVALID'))
        )
        .withColumn(
            'checked_at',
            F.current_timestamp()
        )
    )

In [0]:
def validate_transactions(df_dedup, df_customers, df_franchises, tolerance=0.01):
    return(
        df_dedup
        .join(
            df_customers.select('customerID').distinct().withColumn('customer_exists', F.lit(True)),
            on='customerID',
            how='left'
        )
        .join(
            df_franchises.select('franchiseID').distinct().withColumn('franchise_exists', F.lit(True)),
            on='franchiseID',
            how='left'
        )
        .withColumn(
            'dq_reason',
            # Required keys and product name 
            F.coalesce(
                is_missing('transactionID'),
                is_missing('customerID'),
                is_missing('franchiseID'),
                is_missing('product'), 
            # Date must be convertible and non-null
                F.when(F.col('dateTime').isNull()
                    | F.col('transaction_date').isNull(),
                    F.lit('invalid_date')
                )
                # Quantity > 0
                .when(F.col('quantity').isNull()
                    | (F.col('quantity') <= 0),
                    F.lit('invalid_quantity')
                )
                # Prices >= 0
                .when(F.col('unitPrice').isNull()
                    | (F.col('unitPrice') < 0)
                    | F.col('totalPrice').isNull()
                    | (F.col('totalPrice') < 0),
                    F.lit('invalid_price')
                )
                # Difference greater than tolerance
                .when(F.abs(
                        F.col('totalPrice')
                        - F.col('calculated_total')
                    ) > tolerance,
                    F.lit('price_mismatch')
                )
                # Unknown customers and unknown franchises
                .when(
                    F.col('customer_exists').isNull(),
                    F.lit('unknown_customer')
                )
                .when(
                    F.col('franchise_exists').isNull(),
                    F.lit('unknown_franchise')
                ).otherwise(F.lit(None))
            )
        )
        .withColumn(
            'dq_status',
            F.when(
                F.col('dq_reason').isNull(),
                F.lit('VALID')
            ).otherwise(F.lit('INVALID'))
        )
        .withColumn(
            'checked_at',
            F.current_timestamp()
        )
    )

In [0]:
def split_valid_invalid(df:DataFrame) -> tuple[DataFrame, DataFrame]:
    valid_df = df.filter(
        F.col('dq_status') == 'VALID'
    )
    invalid_df = df.filter(
        F.col('dq_status') == 'INVALID'
    )

    return valid_df, invalid_df
